In [ ]:
from torch import nn, torch
from tiktoken import encoding_for_model
import math

In [ ]:
enc = encoding_for_model("gpt-2")

In [ ]:
BATCH_SIZE = 8
CONTEXT_SIZE = 2048

EMBEDDING_DIM = 512
KEY_MATRIX_DIM = 128
QUERY_MATRIX_DIM = KEY_MATRIX_DIM
VALUE_MATRIX_DIM = 256

token_embedding_table = nn.Embedding(enc.n_vocab, EMBEDDING_DIM)
pos_embedding_table = nn.Embedding(CONTEXT_SIZE, EMBEDDING_DIM)

In [ ]:
class SelfAttention(nn.Module):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.W_Q = nn.Linear(EMBEDDING_DIM, QUERY_MATRIX_DIM)
        self.W_K = nn.Linear(EMBEDDING_DIM, KEY_MATRIX_DIM)
        self.W_V = nn.Linear(EMBEDDING_DIM, VALUE_MATRIX_DIM)
        self.W_out = nn.Linear(VALUE_MATRIX_DIM, EMBEDDING_DIM)

        mask = torch.triu(torch.ones(CONTEXT_SIZE, CONTEXT_SIZE), diagonal=1).bool()

        self.register_buffer("mask", mask)

    def forward(self, x: torch.Tensor):
        # x = dim(batch, context len, embedding dim)

        q: torch.Tensor = self.W_Q(x) # (batch, context len, query dim)
        k: torch.Tensor = self.W_K(x) # (batch, context len, key dim)
        v: torch.Tensor = self.W_V(x) # (batch, context len, value dim)

        scores = q @ torch.transpose(k, 1, 2) / math.sqrt(QUERY_MATRIX_DIM) # (batch, context len (query), context len(key))
        scores = scores.masked_fill_(self.mask, float("-inf")) # (batch, context len(query), context len(key))

        weights = torch.softmax(scores, dim=-1) # (batch, context len(query), context len(key))

        output = weights @ v # (batch, context len, value dim)

        return self.W_out(output)

In [ ]:
model = SelfAttention()

random_token_entry = torch.randn(BATCH_SIZE, CONTEXT_SIZE, EMBEDDING_DIM)

random_token_entry.shape

torch.Size([8, 2048, 512])

In [ ]:
model.forward(random_token_entry).shape

torch.Size([8, 2048, 512])

In [ ]:
model.mask

tensor([[False,  True,  True,  ...,  True,  True,  True],
        [False, False,  True,  ...,  True,  True,  True],
        [False, False, False,  ...,  True,  True,  True],
        ...,
        [False, False, False,  ..., False,  True,  True],
        [False, False, False,  ..., False, False,  True],
        [False, False, False,  ..., False, False, False]])